# Day 1 · Exercise 4: Still Image to Video

**What you'll build:** `image_to_video(image_path, duration, output_path, fps=25)` — converts a still PNG image into a video clip of exactly the specified duration.

**Why it matters:** This is the single-slide version of what `lesson_build.py` does across an entire lesson. Every slide in this course starts as a PNG that becomes a timed video segment. Understanding how to convert a still image to video is a foundational FFmpeg skill.

**How to complete this exercise:**
1. Read the docstring — note the required flags and their purpose
2. Replace `pass` with your implementation using `subprocess.run`
3. Run the **Check Your Work** cell — all 5 checks must pass

> **Hint if you're stuck:** `ffmpeg -loop 1 -i image.png -t 3 -c:v libx264 -pix_fmt yuv420p -r 25 out.mp4`

## Your Implementation

In [ ]:
import subprocess

def image_to_video(
    image_path: str,
    duration: float,
    output_path: str,
    fps: int = 25,
) -> None:
    """
    Convert a still image to a video clip of the specified duration.

    Args:
        image_path:  Path to the input PNG or JPG image.
        duration:    Length of the output video in seconds.
        output_path: Where to save the MP4.
        fps:         Frames per second (default 25).

    Required FFmpeg flags:
        -loop 1         Tell FFmpeg to loop the still image as a video source.
        -t {duration}   Cut the output at exactly this many seconds.
        -c:v libx264    Encode as H.264.
        -pix_fmt yuv420p  Broadest device compatibility.
        -r {fps}        Set the output frame rate.
        -y              Overwrite output without asking.

    The output resolution must match the input image.
    Do not add any scale filter — let FFmpeg use the image dimensions as-is.

    Example:
        image_to_video("slide_001.png", 45.3, "slide_001.mp4")
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    pass
    # ────────────────────────────────────────────────────────────────────────

## Check Your Work

Run the cell below. It generates a test image, calls your function, then uses ffprobe to verify the output.

In [ ]:
import os, subprocess, json
from PIL import Image

_PASS      = '\u2705'
_FAIL      = '\u274c'
_TEST_IMG  = '__check_slide.png'
_TEST_VID  = '__check_slide.mp4'
_W, _H     = 640, 360
_DURATION  = 3.0
_FPS       = 25

def _make_test_image():
    img = Image.new("RGB", (_W, _H), (15, 23, 42))
    img.save(_TEST_IMG)
    return os.path.exists(_TEST_IMG)

def _ffprobe(path):
    r = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_streams", path],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        return None
    data = json.loads(r.stdout)
    vs = next((s for s in data["streams"] if s["codec_type"] == "video"), None)
    if not vs:
        return None
    num, den = map(int, vs["r_frame_rate"].split("/"))
    return {
        "duration": float(vs.get("duration", 0)),
        "width":    int(vs["width"]),
        "height":   int(vs["height"]),
        "fps":      round(num / den, 3),
        "codec":    vs["codec_name"],
    }

def _run_checks():
    score = 0
    total = 5

    if not _make_test_image():
        print(f'{_FAIL} Could not create test image — is Pillow installed?')
        return

    # Check 1: function is callable
    try:
        assert callable(image_to_video)
        print(f'{_PASS} Check 1/5: function exists and is callable')
        score += 1
    except AssertionError:
        print(f'{_FAIL} Check 1/5: image_to_video is not defined')
        return

    # Check 2: creates a file
    try:
        image_to_video(_TEST_IMG, _DURATION, _TEST_VID, fps=_FPS)
        assert os.path.exists(_TEST_VID), f'No file created at {_TEST_VID}'
        print(f'{_PASS} Check 2/5: creates a file at output_path')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 2/5: {e}')

    # Check 3: correct resolution
    info = _ffprobe(_TEST_VID)
    try:
        assert info is not None, 'ffprobe could not read the output file'
        assert info['width']  == _W, f"width: expected {_W}, got {info['width']}"
        assert info['height'] == _H, f"height: expected {_H}, got {info['height']}"
        print(f"{_PASS} Check 3/5: correct resolution ({info['width']}x{info['height']})")
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 3/5: {e}')

    # Check 4: correct fps
    try:
        assert info is not None
        assert abs(info['fps'] - _FPS) < 1.0, \
            f"fps: expected ~{_FPS}, got {info['fps']} — did you pass -r {_FPS}?"
        print(f"{_PASS} Check 4/5: correct fps ({info['fps']})")
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 4/5: {e}')

    # Check 5: correct duration (within 0.5s)
    try:
        assert info is not None
        assert abs(info['duration'] - _DURATION) < 0.5, (
            f"duration: expected ~{_DURATION}s, got {info['duration']:.2f}s — "
            f"did you pass -t {_DURATION}?"
        )
        print(f"{_PASS} Check 5/5: correct duration ({info['duration']:.2f}s)")
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 5/5: {e}')

    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 4 complete! All {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} checks passed. Keep going — you\'re close!')

    for f in [_TEST_IMG, _TEST_VID]:
        if os.path.exists(f):
            os.remove(f)

_run_checks()

## Bonus Challenge

Once all 5 checks pass, try building a two-slide video using your function and the concat demuxer:

1. Create two PNG images (different background colours)
2. Convert each to a video clip using `image_to_video` — different durations
3. Write a concat text file listing both clips
4. Use FFmpeg to join them: `ffmpeg -f concat -safe 0 -i concat.txt -c copy output.mp4`
5. Verify the total duration with ffprobe

This is exactly what `lesson_build.py` does for a 9-slide lesson.

In [ ]:
# Bonus: build a 2-slide video
# from PIL import Image

# img1 = Image.new("RGB", (640, 360), (15, 23, 42))   # dark navy
# img2 = Image.new("RGB", (640, 360), (30, 41, 59))   # slightly lighter
# img1.save("bonus_slide1.png")
# img2.save("bonus_slide2.png")

# image_to_video("bonus_slide1.png", 3.0, "bonus_clip1.mp4")
# image_to_video("bonus_slide2.png", 5.0, "bonus_clip2.mp4")

# with open("bonus_concat.txt", "w") as f:
#     f.write("file 'bonus_clip1.mp4'\n")
#     f.write("file 'bonus_clip2.mp4'\n")

# subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0",
#                 "-i", "bonus_concat.txt", "-c", "copy", "bonus_final.mp4"])
# print("Done — check bonus_final.mp4")

---
## Solution

<details>
<summary>Click to reveal solution — try on your own first</summary>

```python
def image_to_video(
    image_path: str,
    duration: float,
    output_path: str,
    fps: int = 25,
) -> None:
    subprocess.run(
        [
            "ffmpeg", "-y",
            "-loop", "1",
            "-i", image_path,
            "-t", str(duration),
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-r", str(fps),
            output_path,
        ],
        capture_output=True,
        check=True,
    )
```

**Why each flag is there:**
- `-loop 1` — without this, FFmpeg reads the image once (one frame) and stops. The loop makes it a continuous video source.
- `-t {duration}` — cuts the output at exactly this length. Without it, the looped image would run forever.
- `-c:v libx264` — H.264 is the universally supported video codec.
- `-pix_fmt yuv420p` — H.264 supports other pixel formats, but yuv420p is the only one that plays on every device and browser. Skipping this can cause playback failures on mobile or older hardware.
- `-r {fps}` — enforces the frame rate. FFmpeg may choose a strange default frame rate for still image inputs without this.
- `capture_output=True` — silences FFmpeg's verbose progress output in the notebook.
- `check=True` — raises an exception if FFmpeg returns a non-zero exit code, so errors surface immediately.

**What the concat demuxer does differently:**  
`image_to_video` handles a single slide. The concat demuxer handles a list of slides, each with its own duration. `lesson_build.py` uses the concat demuxer directly (not `image_to_video`) because it's more efficient — it reads the PNG files directly without creating intermediate MP4 clips for each slide.

</details>